<a href="https://colab.research.google.com/github/myazzeh/NLP-Course/blob/main/NLP_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install important Libraries
!pip install -U langchain-community --quiet
!pip install langchain --quiet
!pip install langchain_google_genai --quiet
!pip install pypdf --quiet
!pip install chromadb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.5/305.5 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... do

In [ ]:
#Load Dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/resume-dataset


In [ ]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings # Corrected import
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain.llms import OpenAI

# # Set your OpenAI API key
# import getpass
# import os
# from google.colab import userdata
# os.environ["GOOGLE_API_KEY"] =userdata.get('GOOGLE_API_KEY')

# if not os.environ.get("GOOGLE_API_KEY"):
#   os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

# from langchain.chat_models import init_chat_model

# LLM = init_chat_model("gemini-2.0-flash", model_provider="google_genai")
# response = LLM.invoke("how to reduce LDL cholestrol?")
# response.content # Removed the print statement to avoid extra output



#**Process One File**

In [ ]:
filename= '34198885.pdf'
# Paths
PDF_FOLDER = path +"/data/data/ACCOUNTANT/"

# Semantic labels to segment into
CATEGORIES = ["skills", "education", "experience"]

# Create embedding function
embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # Corrected class and model name

# To collect final results
segmented_data = []

# Process each PDF file
#for folder in os.listdir(PDF_FOLDER):
#  print(folder)
#for filename in os.listdir(PDF_FOLDER):
# print(filename)
# if filename.endswith(".pdf"):
#     print(f"Processing {filename}")

# Load PDF
loader = PyPDFLoader(os.path.join(PDF_FOLDER, filename))
pages = loader.load()
# print(pages)
# Split text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = []
for page in pages:
    splits = text_splitter.split_text(page.page_content)
    for s in splits:
        chunks.append(Document(page_content=s))

chunks

[Document(metadata={}, page_content="ACCOUNTANT I\nSummary\nFlexible A\n ccountant who adapts seamlessly to constantly evolving accounting processes and technologies.\nHighlights\nStrong communication skills\nEffective time management\nAnalytical reasoning\nDetail-oriented\nAccount reconciliations\nCustomer-oriented\nFlexible team player\nSuperior research skills\nExperience\nAccountant I\n \n08/2014\n \nto \nCurrent\n \nCompany Name\n \nCity\n \n, \nState\nSet up new jobs and new hires in the Profitool accounting software. \nPrepare weekly invoices and perform research to resolve billing/payroll issues.\nCollect on aged receivables and report to management on a monthly basis. \nPerform reconciliation of accounts and make necessary entries and\nadjustments. \nPerform accounting analysis and conduct special accounting related projects at management's request. \nExamine accounting\ndocuments to verify completeness and conformance with specific accounting requirements. \nTrace and reconci

In [ ]:
# Create vector store in-memory to search
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=None # In-memory
)

# For each category, do a similarity search to find relevant chunks
file_data = {"file": filename}
for category in CATEGORIES:
    query = f"Find parts of this resume that discuss {category}."
    docs = vectordb.similarity_search(query, k=3)
    content = "\n\n".join([doc.page_content for doc in docs])
    file_data[category] = content

segmented_data.append(file_data)

# Example: print the extracted segments
for data in segmented_data:
    print("\n\n=============================")
    print(f"File: {data['file']}")
    print("\n--- Skills ---")
    print(data.get("skills", "Not found"))
    print("\n--- Education ---")
    print(data.get("education", "Not found"))
    print("\n--- Experience ---")
    print(data.get("experience", "Not found"))

#**Process Multiple Files**

In [ ]:
# Paths
PDF_FOLDER = path +"/data/data/ACCOUNTANT/"

# Semantic labels to segment into
CATEGORIES = ["skills", "education", "experience"]

# Create embedding function
embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # Corrected class and model name

# To collect final results
segmented_data = []

# Process each PDF file
#for folder in os.listdir(PDF_FOLDER):
#  print(folder)
for filename in os.listdir(PDF_FOLDER):
    print(filename)
    if filename.endswith(".pdf"):
        print(f"Processing {filename}")

        # Load PDF
        loader = PyPDFLoader(os.path.join(PDF_FOLDER, filename))
        pages = loader.load()

        # Split text
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100
        )
        chunks = []
        for page in pages:
            splits = text_splitter.split_text(page.page_content)
            for s in splits:
                chunks.append(Document(page_content=s))

        # Create vector store in-memory to search
        vectordb = Chroma.from_documents(
            documents=chunks,
            embedding=embedding,
            persist_directory=None # In-memory
        )

        # For each category, do a similarity search to find relevant chunks
        file_data = {"file": filename}
        for category in CATEGORIES:
            query = f"Find parts of this resume that discuss {category}."
            docs = vectordb.similarity_search(query, k=3)
            content = "\n\n".join([doc.page_content for doc in docs])
            file_data[category] = content

        segmented_data.append(file_data)

# Example: print the extracted segments
for data in segmented_data:
    print("\n\n=============================")
    print(f"File: {data['file']}")
    print("\n--- Skills ---")
    print(data.get("skills", "Not found"))
    print("\n--- Education ---")
    print(data.get("education", "Not found"))
    print("\n--- Experience ---")
    print(data.get("experience", "Not found"))

27558837.pdf
Processing 27558837.pdf


ERROR:grpc._plugin_wrapping:AuthMetadataPluginCallback "<google.auth.transport.grpc.AuthMetadataPlugin object at 0x79e50297f410>" raised exception!
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/auth/compute_engine/credentials.py", line 126, in refresh
    self._retrieve_info(request)
  File "/usr/local/lib/python3.11/dist-packages/google/auth/compute_engine/credentials.py", line 99, in _retrieve_info
    info = _metadata.get_service_account_info(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/google/auth/compute_engine/_metadata.py", line 338, in get_service_account_info
    return get(request, path, params={"recursive": "true"})
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/google/auth/compute_engine/_metadata.py", line 263, in get
    raise exceptions.TransportError(
google.auth.exceptions.TransportError: ("Failed to retrieve http:/

GoogleGenerativeAIError: Error embedding content: Timeout of 60.0s exceeded, last exception: 503 Getting metadata from plugin failed with error: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google.auth.transport.requests._Response object at 0x79e501e634d0>)